# Attention Mimarileri: Encoder, Decoder ve Set Transformer

Bu notebook, **normal/self/cross attention** farklarını; **encoder-only**, **encoder-decoder**, **decoder-only** ve **Set Transformer** mimarileri üzerinden karşılaştırmalı açıklar.


## 1. Temel ayrım

```text
Self-attention:
Q, K, V aynı yapıdan gelir.

Cross-attention:
Q bir taraftan, K ve V başka taraftan gelir.

Set Transformer:
Sequence yerine set işler; positional encoding genellikle yoktur.
```

Attention formülü:

$$
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Burada:

| Terim | Sezgisel anlam |
|---|---|
| Query / Q | Ben ne arıyorum? |
| Key / K | Bende hangi bilgi var? |
| Value / V | Taşınacak asıl bilgi ne? |


## 2. Encoder-only Transformer

BERT benzeri mimarilerde kullanılır. Temel olarak sadece **self-attention** vardır.

```text
Input tokens
["Ali", "kitabı", "okudu"]

        ↓

Token Embedding + Positional Encoding

        ↓

┌─────────────────────────────┐
│ Encoder Layer 1             │
│                             │
│  Multi-Head Self-Attention  │
│       Q = X                 │
│       K = X                 │
│       V = X                 │
│                             │
│  Add & Norm                 │
│  Feed Forward               │
│  Add & Norm                 │
└─────────────────────────────┘

        ↓

┌─────────────────────────────┐
│ Encoder Layer 2             │
│                             │
│  Multi-Head Self-Attention  │
│       Q = H1                │
│       K = H1                │
│       V = H1                │
│                             │
│  Add & Norm                 │
│  Feed Forward               │
│  Add & Norm                 │
└─────────────────────────────┘

        ↓

Encoder hidden states
```

Burada her token aynı cümledeki diğer tokenlara bakar:

```text
"Ali"    → "kitabı", "okudu"
"kitabı" → "Ali", "okudu"
"okudu"  → "Ali", "kitabı"
```


## 3. Encoder-decoder Transformer

Bu yapı çeviri, özetleme ve text-to-text modellerinde görülür. Örneğin klasik Transformer, T5 ve BART gibi modeller bu aileye yakındır.

Kaynak cümle:

```text
Ali kitabı okudu.
```

Hedef cümle:

```text
Ali read the book.
```

Mimari genel olarak şöyle:

```text
SOURCE SIDE                                TARGET SIDE

Input tokens                               Target tokens so far
["Ali", "kitabı", "okudu"]                ["<BOS>", "Ali", "read", "the"]

      ↓                                             ↓

Embedding + Positional Encoding             Embedding + Positional Encoding

      ↓                                             ↓

┌─────────────────────────────┐              ┌─────────────────────────────┐
│ Encoder Layer 1             │              │ Decoder Layer 1             │
│                             │              │                             │
│ Self-Attention              │              │ Masked Self-Attention       │
│ Q = X_enc                   │              │ Q = X_dec                   │
│ K = X_enc                   │              │ K = X_dec                   │
│ V = X_enc                   │              │ V = X_dec                   │
│                             │              │                             │
│ Feed Forward                │              │ Cross-Attention             │
│                             │              │ Q = decoder hidden states   │
└─────────────────────────────┘              │ K = encoder hidden states   │
                                             │ V = encoder hidden states   │
      ↓                                      │                             │
                                             │ Feed Forward                │
┌─────────────────────────────┐              └─────────────────────────────┘
│ Encoder Layer 2             │
│                             │                       ↓
│ Self-Attention              │              ┌─────────────────────────────┐
│ Q = H_enc                   │              │ Decoder Layer 2             │
│ K = H_enc                   │              │                             │
│ V = H_enc                   │              │ Masked Self-Attention       │
│                             │              │ Cross-Attention             │
│ Feed Forward                │              │ Feed Forward                │
└─────────────────────────────┘              └─────────────────────────────┘
      ↓                                               ↓
Encoder hidden states                         Vocabulary projection
                                                      ↓
                                             Next token probabilities
                                                      ↓
                                             Generated token
```


## 4. Encoder-decoder içinde attention türleri

| Bölüm | Attention tipi | Q nereden gelir? | K/V nereden gelir? |
|---|---|---|---|
| Encoder | Self-attention | Encoder input/hidden state | Encoder input/hidden state |
| Decoder ilk attention | Masked self-attention | Decoder hidden state | Decoder hidden state |
| Decoder ikinci attention | Cross-attention | Decoder hidden state | Encoder hidden state |

Decoder layer içindeki sıralama:

```text
Target hidden states
        │
        ↓
┌────────────────────────────────────┐
│ 1. Masked Multi-Head Self-Attention │
│                                    │
│ Q = target hidden states           │
│ K = target hidden states           │
│ V = target hidden states           │
└────────────────────────────────────┘
        │
        ↓
Add & Norm
        │
        ↓
┌────────────────────────────────────┐
│ 2. Multi-Head Cross-Attention       │
│                                    │
│ Q = decoder hidden states          │
│ K = encoder hidden states          │
│ V = encoder hidden states          │
└────────────────────────────────────┘
        │
        ↓
Add & Norm
        │
        ↓
┌────────────────────────────────────┐
│ 3. Feed Forward Network             │
└────────────────────────────────────┘
        │
        ↓
Add & Norm
        │
        ↓
Decoder layer output
```


## 5. Cross-attention sezgisi

Decoder masked self-attention'dan sonra şuna benzer bir bilgiye sahiptir:

```text
Şu ana kadar hedefte şunu yazdım:
"Ali read the ..."
```

Şimdi decoder kaynak cümleye bakar:

```text
Kaynak cümlede bu pozisyon için hangi bilgi önemli?
```

Cross-attention burada devreye girer:

```python
Q = decoder_hidden_states @ Wq
K = encoder_hidden_states @ Wk
V = encoder_hidden_states @ Wv
```

Örneğin `book` üretilecek pozisyonda:

```text
Decoder query:
"Ben şu an isim/nesne bekliyorum."

Encoder keys:
Ali      → düşük skor
kitabı   → yüksek skor
okudu    → düşük skor
```

Attention ağırlıkları şöyle olabilir:

```text
Ali      0.05
kitabı   0.85
okudu    0.10
```

Sonra encoder value'ları ağırlıklı toplanır:

```text
context = 0.05 * V_Ali + 0.85 * V_kitabı + 0.10 * V_okudu
```

Bu context decoder'a kaynak tarafta ilgili bilginin `kitabı` tokenında olduğunu söyler. Sonra vocabulary projection ile `book` tokenı yüksek olasılık alır.


## 6. Encoder-only, decoder-only, encoder-decoder karşılaştırması

| Mimari | Örnek model | Attention tipi | Kullanım |
|---|---|---|---|
| Encoder-only | BERT | Bidirectional self-attention | Classification, embedding, NER |
| Decoder-only | GPT | Causal/masked self-attention | Text generation |
| Encoder-decoder | T5, BART, klasik Transformer | Encoder self-attention + decoder masked self-attention + cross-attention | Translation, summarization, text-to-text |

Decoder-only GPT tarzında normalde encoder olmadığı için cross-attention yoktur.

```text
GPT-style decoder-only:

Input tokens
   ↓
Masked Self-Attention
   ↓
Feed Forward
   ↓
Next token prediction
```

Encoder-decoder'da ise decoder kaynak tarafına da bakar:

```text
Source tokens → Encoder → Encoder hidden states
                                      ↓
Target tokens → Decoder self-attn → Cross-attn → Next token
```


## 7. Set Transformer

Set Transformer, sequence değil **set** işler.

Sequence örneği:

```text
["Ali", "kitabı", "okudu"]
```

Burada sıra önemlidir.

Set örneği:

```text
{x1, x2, x3, ..., xn}
```

Burada sıra önemli değildir. Örneğin:

```text
{müşteri_1, müşteri_2, müşteri_3}
```

ile

```text
{müşteri_3, müşteri_1, müşteri_2}
```

aynı set'tir. Bu yüzden Set Transformer'da genellikle positional encoding kullanılmaz.


## 8. Set Transformer'daki temel blok: MAB

Set Transformer'ın ana bloğu:

```text
MAB(X, Y)
```

Açılımı: **Multihead Attention Block**.

Mantığı:

```text
Q = X
K = Y
V = Y
```

Yani:

```python
Q = X @ Wq
K = Y @ Wk
V = Y @ Wv
```

Şema:

```text
X ───────────────→ Q
                   │
                   ↓
             Attention(Q, K, V)
                   ↑
                   │
Y ─────→ K
Y ─────→ V
```

MAB iki farklı input alabilir:

```text
MAB(X, X) → self-attention
MAB(X, Y) → cross-attention benzeri
```


## 9. Set Transformer blokları

| Blok | Açılım | Mantık |
|---|---|---|
| MAB(X, Y) | Multihead Attention Block | X, Y'ye attention verir |
| SAB(X) | Set Attention Block | MAB(X, X), yani self-attention |
| ISAB(X) | Induced Set Attention Block | Büyük setlerde attention maliyetini düşürür |
| PMA(X) | Pooling by Multihead Attention | Set'i sabit sayıda output'a indirger |


## 10. SAB: Set Attention Block

SAB şudur:

```text
SAB(X) = MAB(X, X)
```

Yani self-attention'dır.

```text
Input set X
{x1, x2, x3, x4}

        ↓

┌─────────────────────────────┐
│ SAB                         │
│                             │
│ MAB(X, X)                   │
│                             │
│ Q = X                       │
│ K = X                       │
│ V = X                       │
│                             │
│ Her eleman diğer elemanlara │
│ bakar                       │
└─────────────────────────────┘

        ↓

Updated set representations
```


## 11. ISAB: Induced Set Attention Block

SAB'in maliyeti yüksektir. Eğer set boyutu `n` ise self-attention maliyeti yaklaşık `O(n²)` olur. Çünkü her eleman her elemana bakar.

ISAB bunu azaltmak için **inducing points** kullanır. Inducing points öğrenilebilir vektörlerdir.

```text
I = {i1, i2, ..., im}
```

Genellikle:

```text
m << n
```

ISAB iki attention aşamasından oluşur:

```text
ISAB(X) = MAB(X, H)
where H = MAB(I, X)
```

Şema:

```text
Input set X
{x1, x2, x3, ..., xn}

Learnable inducing points I
{i1, i2, ..., im}

        ↓

Step 1:
┌─────────────────────────────┐
│ MAB(I, X)                   │
│                             │
│ Q = I                       │
│ K = X                       │
│ V = X                       │
│                             │
│ Inducing points input set'ten│
│ özet bilgi çeker            │
└─────────────────────────────┘

        ↓

H = induced summaries

        ↓

Step 2:
┌─────────────────────────────┐
│ MAB(X, H)                   │
│                             │
│ Q = X                       │
│ K = H                       │
│ V = H                       │
│                             │
│ Her set elemanı özetlere    │
│ bakar                       │
└─────────────────────────────┘

        ↓

Output set representations
```

Bu maliyeti yaklaşık `O(nm)` seviyesine indirir. Eğer `m` küçükse çok daha verimlidir.


## 12. PMA: Pooling by Multihead Attention

SAB ve ISAB set elemanlarını günceller ama output hâlâ set boyutundadır.

```text
X = {x1, x2, ..., xn}
Z = {z1, z2, ..., zn}
```

Classification/regression için tek bir vektör gerekebilir. Klasik pooling seçenekleri şunlardır:

```text
mean pooling
sum pooling
max pooling
```

Set Transformer ise attention tabanlı pooling yapar.

```text
PMA(X) = MAB(S, X)
```

Burada `S`, öğrenilebilir seed vector'lerdir.

```text
S = {s1, s2, ..., sk}
```

Şema:

```text
Learnable seed vectors S
{s1, ..., sk}

        ↓ Q

┌─────────────────────────────┐
│ PMA                         │
│                             │
│ MAB(S, X)                   │
│                             │
│ Q = S                       │
│ K = X                       │
│ V = X                       │
│                             │
│ Seed vector'ler set içinden │
│ bilgi toplar                │
└─────────────────────────────┘

        ↓

k adet pooled output
```

Eğer `k = 1` ise bütün set tek vektöre iner. Eğer `k > 1` ise birkaç prototip/cluster-like output üretilebilir.


## 13. Set Transformer genel mimari

Set Transformer genelde encoder-decoder gibi iki parçaya ayrılır, ama buradaki decoder text decoder gibi token üretmek zorunda değildir.

```text
Input set X
{x1, x2, ..., xn}

        ↓

┌─────────────────────────────┐
│ Set Encoder                 │
│                             │
│ SAB or ISAB                 │
│ SAB or ISAB                 │
│ SAB or ISAB                 │
└─────────────────────────────┘

        ↓

Contextualized set representations

        ↓

┌─────────────────────────────┐
│ Set Decoder                 │
│                             │
│ PMA                         │
│ SAB                         │
│ Feed Forward / Output Head  │
└─────────────────────────────┘

        ↓

Prediction
```

Daha kısa hali:

```text
Input set X
        ↓
ISAB/SAB
        ↓
ISAB/SAB
        ↓
ISAB/SAB
        ↓
PMA
        ↓
SAB
        ↓
Linear head
        ↓
Output
```


## 14. Set Transformer ile encoder-decoder Transformer karşılaştırması

| Özellik | Encoder-decoder Transformer | Set Transformer |
|---|---|---|
| Veri tipi | Sequence | Set |
| Sıra önemli mi? | Evet | Hayır |
| Positional encoding | Genelde var | Genelde yok |
| Encoder | Source tokenları işler | Set elemanlarını işler |
| Decoder | Target token üretir | Pooling / set-level output üretir |
| Cross-attention | Decoder → Encoder | PMA veya MAB(X, Y) |
| Output | Token sequence | Set-level veya element-level prediction |
| Örnek görev | Çeviri, özetleme | Set classification, point cloud, clustering, multiple instance learning |


## 15. Attention kaynakları karşılaştırmalı tablo

| Yapı | Form | Q | K | V | Yorum |
|---|---|---|---|---|---|
| Encoder self-attention | Attention(X, X, X) | Encoder hidden | Encoder hidden | Encoder hidden | Kaynak tokenlar birbirine bakar |
| Decoder masked self-attention | Attention(Y, Y, Y) | Decoder hidden | Decoder hidden | Decoder hidden | Hedef tokenlar geçmişe bakar |
| Decoder cross-attention | Attention(Y, X, X) | Decoder hidden | Encoder hidden | Encoder hidden | Hedef taraf kaynak tarafına bakar |
| SAB | MAB(X, X) | Set X | Set X | Set X | Set içi self-attention |
| ISAB step 1 | MAB(I, X) | Inducing points | Set X | Set X | Özet noktalar set'ten bilgi toplar |
| ISAB step 2 | MAB(X, H) | Set X | Induced summaries | Induced summaries | Set elemanları özetlerden bilgi alır |
| PMA | MAB(S, X) | Seed vectors | Set X | Set X | Seed vectors set'i pool eder |


## 16. Üç mimariyi yan yana şema

```text
A) Encoder-only Transformer

Input sequence X
      ↓
Embedding + Position
      ↓
Self-Attention
Q=X, K=X, V=X
      ↓
Feed Forward
      ↓
Hidden states
      ↓
Task head


B) Encoder-decoder Transformer

Source X                         Target Y
   ↓                                ↓
Embedding + Position               Embedding + Position
   ↓                                ↓
Encoder Self-Attention             Masked Self-Attention
Q=X,K=X,V=X                        Q=Y,K=Y,V=Y
   ↓                                ↓
Encoder hidden states  ───────→    Cross-Attention
                                   Q=Y_hidden
                                   K=Encoder_hidden
                                   V=Encoder_hidden
                                    ↓
                                   Feed Forward
                                    ↓
                                   Vocabulary head
                                    ↓
                                   Generated token


C) Set Transformer

Input set X
{x1, x2, ..., xn}
      ↓
No positional encoding generally
      ↓
SAB or ISAB
SAB:  Q=X,K=X,V=X
ISAB: Q=I,K=X,V=X then Q=X,K=H,V=H
      ↓
Set representations
      ↓
PMA
Q=Seed vectors
K=Set representations
V=Set representations
      ↓
Set-level output
```


## 17. Set Transformer'da permutation kavramları

Normal Transformer'da sıra önemlidir:

```text
"Ali kitabı okudu"
```

ile

```text
"okudu Ali kitabı"
```

aynı değildir. Bu yüzden positional encoding gerekir.

Set Transformer'da sıra önemsizdir:

```text
{x1, x2, x3}
```

ile

```text
{x3, x1, x2}
```

aynı input kabul edilir.

### Permutation equivariant

Input sırası değişirse output sırası da aynı şekilde değişir.

```text
Input:
[x1, x2, x3]

Output:
[z1, z2, z3]

Input reorder:
[x3, x1, x2]

Output reorder:
[z3, z1, z2]
```

SAB/ISAB gibi set encoder blokları genelde buna uygundur.

### Permutation invariant

Input sırası değişse de final output değişmez.

```text
Input:
[x1, x2, x3]

Output:
class A

Input reorder:
[x3, x1, x2]

Output:
class A
```

PMA + pooling/head ile bu elde edilir.


## 18. Kod karşılığı

```python
# Encoder self-attention
Q = encoder_hidden @ Wq
K = encoder_hidden @ Wk
V = encoder_hidden @ Wv

# Decoder masked self-attention
Q = decoder_hidden @ Wq
K = decoder_hidden @ Wk
V = decoder_hidden @ Wv

# Encoder-decoder cross-attention
Q = decoder_hidden @ Wq
K = encoder_hidden @ Wk
V = encoder_hidden @ Wv

# Set Transformer SAB
Q = X @ Wq
K = X @ Wk
V = X @ Wv

# Set Transformer PMA
Q = S @ Wq       # learnable seed vectors
K = X @ Wk
V = X @ Wv

# Set Transformer ISAB
H = MAB(I, X)    # inducing points attend to set
Z = MAB(X, H)    # set attends to summaries
```


## 19. Kısa özet

```text
Encoder-only:
Her token aynı inputtaki tokenlara bakar.
Sadece self-attention vardır.

Encoder-decoder:
Encoder kaynak cümlede self-attention yapar.
Decoder hedef tarafta masked self-attention yapar.
Decoder sonra encoder çıktısına cross-attention yapar.

Set Transformer:
Input sequence değil set'tir.
SAB = set self-attention.
ISAB = inducing point ile ucuzlatılmış set attention.
PMA = seed vector ile attention-based pooling.
```

En iyi mental model:

```text
Self-attention:
Ben kendi grubumdaki elemanlara bakıyorum.

Cross-attention:
Ben başka bir grubun elemanlarına bakıyorum.

Set Transformer:
Ben sırası olmayan bir gruptaki elemanlar arasında attention kuruyorum.
```
